# 文を一階述語論理の式に変換して真偽を確かめる (NLTK)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/10_semantics/logic_semantics.ipynb)

NLP クックブック 10-11 ／ 意味解析

「すべての少女が犬を追いかける」のような文の意味を、論理式として厳密に表したい。モデル (世界の状態) の上で真偽を計算し、推論もしたい。

In [1]:
import nltk
nltk.download("book_grammars", quiet=True)

# 各規則に意味表現 (ラムダ式) が付いた素性文法。構文木を下から合成して、文全体の論理式を得る
parser = nltk.load_parser("grammars/book_grammars/simple-sem.fcfg", trace=0)
for sent in ["Irene walks", "a dog barks", "Angus chases a dog", "every girl chases a dog", "Angus sees Irene"]:
    for tree in parser.parse(sent.split()):
        print(f"{sent:26} {tree.label()['SEM']}")

Irene walks                walk(irene)
a dog barks                exists x.(dog(x) & bark(x))
Angus chases a dog         exists z1.(dog(z1) & chase(angus,z1))
every girl chases a dog    all x.(girl(x) -> exists z2.(dog(z2) & chase(x,z2)))
Angus sees Irene           see(angus,irene)


In [2]:
# 文法の中身を見る: 限定詞や動詞に対応するラムダ式
print(nltk.data.load("grammars/book_grammars/simple-sem.fcfg", format="text")[430:1300])

t(?nom)> ] -> Det[NUM=?n,SEM=?det]  Nom[NUM=?n,SEM=?nom]
NP[LOC=?l,NUM=?n,SEM=?np] -> PropN[LOC=?l,NUM=?n,SEM=?np]

Nom[NUM=?n,SEM=?nom] -> N[NUM=?n,SEM=?nom]

VP[NUM=?n,SEM=?v] -> IV[NUM=?n,SEM=?v]
VP[NUM=?n,SEM=<?v(?obj)>] -> TV[NUM=?n,SEM=?v] NP[SEM=?obj]
VP[NUM=?n,SEM=<?v(?obj,?pp)>] -> DTV[NUM=?n,SEM=?v] NP[SEM=?obj] PP[+TO,SEM=?pp]

PP[+TO, SEM=?np] -> P[+TO] NP[SEM=?np]

#############################
# Lexical Rules
#############################

PropN[-LOC,NUM=sg,SEM=<\P.P(angus)>] -> 'Angus'
PropN[-LOC,NUM=sg,SEM=<\P.P(cyril)>] -> 'Cyril'
PropN[-LOC,NUM=sg,SEM=<\P.P(irene)>] -> 'Irene'

Det[NUM=sg,SEM=<\P Q.all x.(P(x) -> Q(x))>] -> 'every'
Det[NUM=pl,SEM=<\P Q.all x.(P(x) -> Q(x))>] -> 'all'
Det[SEM=<\P Q.exists x.(P(x) & Q(x))>] -> 'some'
Det[NUM=sg,SEM=<\P Q.exists x.(P(x) & Q(x))>] -> 'a'
Det[NUM=sg,SEM=<\P Q.exists x.(P(x) & Q(x))>] -> 'an'

N[


In [3]:
# モデル (個体の集合と、述語の解釈) を定義して、式の真偽を評価する
val = nltk.Valuation.fromstring("""
    angus => a
    irene => i
    girl => {i, j}
    boy => {a}
    dog => {d1, d2}
    walk => {i, a}
    bark => {d1}
    chase => {(i, d1), (j, d2), (a, d1)}
    see => {(a, i)}
""")
model = nltk.Model(val.domain, val)
g = nltk.Assignment(val.domain)

for sent in ["Irene walks", "every girl chases a dog", "every girl walks", "a dog barks", "Irene sees Angus"]:
    sem = next(parser.parse(sent.split())).label()["SEM"]
    print(f"{model.evaluate(str(sem), g)!s:6} {sent:26} {sem}")

True   Irene walks                walk(irene)
True   every girl chases a dog    all x.(girl(x) -> exists z3.(dog(z3) & chase(x,z3)))
False  every girl walks           all x.(girl(x) -> walk(x))
True   a dog barks                exists x.(dog(x) & bark(x))
False  Irene sees Angus           see(irene,angus)


In [4]:
# 量化子のスコープの曖昧性: 「どの少女も (それぞれ別の) 犬を追う」と「ある 1 匹の犬を全員が追う」
read = nltk.sem.Expression.fromstring
wide_every = read("all x.(girl(x) -> exists y.(dog(y) & chase(x,y)))")
wide_some = read("exists y.(dog(y) & all x.(girl(x) -> chase(x,y)))")
print("∀ が広いスコープ:", model.evaluate(str(wide_every), g), " ∃ が広いスコープ:", model.evaluate(str(wide_some), g))

# 定理証明器で含意関係を確かめる (モデルによらず、論理的に必ず成り立つか)
prover = nltk.ResolutionProver()
print("∃∀ ならば ∀∃:", prover.prove(wide_every, [wide_some]))
print("∀∃ ならば ∃∀:", prover.prove(wide_some, [wide_every]))
premises = [read("all x.(dog(x) -> animal(x))"), read("all x.(animal(x) -> mortal(x))"), read("dog(pochi)")]
print("ポチは死すべきものである:", prover.prove(read("mortal(pochi)"), premises))

∀ が広いスコープ: True  ∃ が広いスコープ: False
∃∀ ならば ∀∃: True
∀∃ ならば ∃∀: False
ポチは死すべきものである: True


- 構成性の原理: 文の意味は、部分の意味とその組み合わせ方から決まる。ラムダ計算は、その「組み合わせ」を関数適用として表す道具。
- `every girl chases a dog` には 2 通りの読みがあるが、この文法が出すのは片方だけ。両方を得るには、Cooper storage などスコープを後から決める仕組みが要る。
- 手書きの文法が扱える文はごく一部。広い範囲の文を論理式にするには、CCG パーザに基づく ccg2lambda や、AMR などの意味表現へのニューラルなパージングが使われる。
- 自然言語推論を「埋め込みの類似度」ではなく「証明」で行うアプローチは、否定や数量を含む推論に強い。